# Stored execution evidence — APTOS 2019 Blindness Detection

This public evidence copy preserves every textual output cell supplied in `jiaozi.zip`.
The original notebook SHA-256 is `dee1375b8cee7e3971f9a8f32487c0d9dd5e368327c0d6c7e1e885634cf34f2a`. Embedded binary rich media is removed so
that competition data and generated images are not redistributed. The repository setup cell
now targets `Isso-W/Jiaozi` at `main`; stored metrics and submission receipts remain historical
outputs and are not presented as a fresh rerun of the current branch.


# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `main` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. Add `KAGGLE_API_TOKEN` in Colab Secrets and accept the Plant Pathology competition rules on Kaggle before downloading.
3. Select a GPU runtime for real training.
4. Run the cells in order.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.fullmatch(r'\[(.*?)\]\((https?://[^)]+)\)', value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r'https?://\S+', value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value

REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'main')
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

print("REPO_URL =", REPO_URL)
print("REPO_REF =", REPO_REF)

clone_cmd = [
    'git', 'clone',
    '--depth', '1',
    '--branch', REPO_REF,
    REPO_URL,
    str(REPO_DIR),
]

completed = subprocess.run(
    clone_cmd,
    capture_output=True,
    text=True,
)

print("=== git stdout ===")
print(completed.stdout)
print("=== git stderr ===")
print(completed.stderr)

completed.check_returncode()

os.chdir(REPO_DIR)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("\n=== requirements.txt ===")
print(Path("requirements.txt").read_text()[:4000])

print("\n=== installing requirements ===")
pip_result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
    capture_output=True,
    text=True,
)

print(pip_result.stdout)
print(pip_result.stderr)
print("pip return code =", pip_result.returncode)

if pip_result.returncode != 0:
    raise RuntimeError("pip install failed. Check the pip output above.")

print('Jiaozi repository:', REPO_DIR)
print('Jiaozi repo URL:', REPO_URL)
print('Jiaozi ref:', REPO_REF)

pipeline_candidates = list(REPO_DIR.rglob("pipeline.py"))
print("pipeline.py candidates:")
for p in pipeline_candidates:
    print(" -", p)


REPO_URL = https://github.com/muzhi-hac/Jiaozi-rag-wang.git
REPO_REF = rag_wang
=== git stdout ===

=== git stderr ===
Cloning into '/content/Jiaozi'...


=== requirements.txt ===
# Runtime dependencies for the Jiaozi CV Auto-DL pipeline.
# Install with the same Python interpreter used to run tests, for example:
#   /usr/bin/python3 -m pip install -r requirements.txt

chromadb
datasets
networkx
numpy
openai>=2.0.0
pandas
pillow
scikit-learn
sentence-transformers
torch
torchvision
transformers

# Test runner for the module4_agent pytest suite.
pytest


=== installing requirements ===
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 129.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 133.5 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-5.5')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token


## Download the Plant Pathology data

These cells configure the Kaggle CLI, download `plant-pathology-2020-fgvc7`, and extract the competition files under `/content/data`. The pipeline cell later converts the one-hot `train.csv` into a single `label` column that Jiaozi's local CSV loader can consume.


In [3]:
import os
import subprocess
import sys
from pathlib import Path
from google.colab import userdata

def run_pip(args):
    print("Running:", " ".join([sys.executable, "-m", "pip"] + args))
    result = subprocess.run(
        [sys.executable, "-m", "pip"] + args,
        capture_output=True,
        text=True,
    )
    print("=== pip stdout ===")
    print(result.stdout)
    print("=== pip stderr ===")
    print(result.stderr)
    print("return code =", result.returncode)
    return result

# First, upgrade the basic installation tools.
run_pip(["install", "--upgrade", "pip", "setuptools", "wheel"])

# Then install the Kaggle CLI; do not use the `-q` flag.
kaggle_install = run_pip(["install", "--upgrade", "kaggle"])

if kaggle_install.returncode != 0:
    raise RuntimeError("Kaggle CLI install failed. Check pip output above.")

# Read the Kaggle access token from Colab Secrets.
kaggle_token = userdata.get("KAGGLE_API_TOKEN")
if not kaggle_token:
    raise RuntimeError("Please add KAGGLE_API_TOKEN in Colab Secrets.")

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)

access_token_path = kaggle_dir / "access_token"
access_token_path.write_text(kaggle_token.strip(), encoding="utf-8")
access_token_path.chmod(0o600)

os.environ["KAGGLE_API_TOKEN"] = kaggle_token.strip()

print("Kaggle access token configured.")

# Check if Kaggle is available.
subprocess.run(["kaggle", "--version"], check=False)


Running: /usr/bin/python3 -m pip install --upgrade pip setuptools wheel
=== pip stdout ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 72.3 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2

=== pip stderr ===
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 83.0.0 which is incompatible.

return code = 0
Running: /usr/bin/python3 -m pip install --upgrade kaggle
=== pip stdout

CompletedProcess(args=['kaggle', '--version'], returncode=0)

In [4]:
KAGGLE_COMPETITION = "aptos2019-blindness-detection"
!kaggle competitions download -c {KAGGLE_COMPETITION} -p /content/data/aptos
!ls -lh /content/data/aptos


100% 9.51G/9.51G [00:52<00:00, 195MB/s]

total 9.6G
-rw-r--r-- 1 root root 9.6G Dec 18  2019 aptos2019-blindness-detection.zip


In [5]:
from pathlib import Path
import zipfile

DATA_ROOT = Path("/content/data/aptos")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

zip_files = sorted(DATA_ROOT.glob(f"{KAGGLE_COMPETITION}*.zip")) or sorted(DATA_ROOT.glob("*.zip"))
print("Zip files:", zip_files)

if not zip_files:
    raise FileNotFoundError("No .zip file found in /content/data/aptos")

zip_path = zip_files[0]
KAGGLE_DATA_DIR = DATA_ROOT / zip_path.stem
KAGGLE_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Extracting:", zip_path)
print("To:", KAGGLE_DATA_DIR)
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(KAGGLE_DATA_DIR)

# Some Kaggle competitions contain nested archives. Extract them in place so image
# discovery below can find the real files regardless of packaging.
for nested_zip in sorted(KAGGLE_DATA_DIR.rglob("*.zip")):
    nested_target = nested_zip.with_suffix("")
    nested_target.mkdir(parents=True, exist_ok=True)
    print("Extracting nested:", nested_zip, "->", nested_target)
    with zipfile.ZipFile(nested_zip, "r") as z:
        z.extractall(nested_target)

print("Done.")
print("KAGGLE_DATA_DIR =", KAGGLE_DATA_DIR)
print("Top-level files:")
for p in sorted(KAGGLE_DATA_DIR.iterdir()):
    print(" -", p.name)


Zip files: [PosixPath('/content/data/aptos/aptos2019-blindness-detection.zip')]
Extracting: /content/data/aptos/aptos2019-blindness-detection.zip
To: /content/data/aptos/aptos2019-blindness-detection
Done.
KAGGLE_DATA_DIR = /content/data/aptos/aptos2019-blindness-detection
Top-level files:
 - sample_submission.csv
 - test.csv
 - test_images
 - train.csv
 - train_images


## Mount Google Drive for caches and checkpoints

HuggingFace caches and training checkpoints can live on Drive so a recycled Colab runtime does not force every artifact to be downloaded or trained from scratch.


In [6]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Everything HuggingFace caches (datasets + hub) lives here, on Drive.
HF_CACHE_DIR = '/content/drive/MyDrive/Jiaozi/hf_cache'
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.environ['HF_HOME'] = HF_CACHE_DIR
os.environ['HF_DATASETS_CACHE'] = os.path.join(HF_CACHE_DIR, 'datasets')

print('HF_HOME =', os.environ['HF_HOME'])
print('First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).')
print('Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.')


Mounted at /content/drive
HF_HOME = /content/drive/MyDrive/Jiaozi/hf_cache
First run downloads the dataset to Drive; later runs reuse it (no 12-min re-download).
Note: reading from Drive (FUSE) is a bit slower than local disk, but far faster than re-downloading.


## Run the integrated pipeline

This cell runs the integrated Jiaozi flow against the local Kaggle files:

1. Convert Plant Pathology's one-hot labels into a local CSV with `image_id,label`.
2. Run Module 1 from the natural-language `QUERY`.
3. Run Module 2 on sampled real images from the Kaggle folder.
4. Run Module 3 retrieval and optional recommender re-ranking.
5. Run Module 4 code generation with local CSV training metadata.

Set `REAL_TRAINING = True` to generate real-training code and train it in the next cell. With `REAL_TRAINING = False`, Module 4 keeps the generated project in smoke-test mode.


In [7]:
# [Full version replacement Cell] APTOS 2019 Kaggle data → Jiaozi Module 1-4

import json
import os
import shutil
import sys
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/Jiaozi")
PIPELINE_PATH = REPO_DIR / "pipeline.py"

if not PIPELINE_PATH.exists():
    raise FileNotFoundError(
        f"No pipeline.py found at {PIPELINE_PATH}. Please rerun the git clone cell first."
    )

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Current working directory:", Path.cwd())
print("Using pipeline:", PIPELINE_PATH)

# -----------------------------
# 1. APTOS task prompt
# -----------------------------
QUERY = (
   "Diabetic retinopathy severity grading from retinal fundus photographs. "
   "Use label_code as the numeric target column for 5 ordered classes. "
   "Evaluate using Quadratic Weighted Kappa. "
   "This is a medical retinal image task, so the pipeline should include robust image preprocessing, "
   "such as resizing, normalization, color and brightness augmentation, and full fine-tuning of a pretrained model. "
   "Prefer EfficientNet-B0 or EfficientNet-B3 with ImageNet pretrained weights. "
   "Handle class imbalance and save the best checkpoint according to validation QWK."

)


REAL_TRAINING = True
EPOCHS = None
OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")

# -----------------------------
# 2. Find Kaggle APTOS data
# -----------------------------
DATASET_DIR = Path(KAGGLE_DATA_DIR)

raw_train_csvs = sorted(DATASET_DIR.rglob("train.csv"))
if not raw_train_csvs:
    raise FileNotFoundError(f"No train.csv found under {DATASET_DIR}")

raw_train_csv = raw_train_csvs[0]
raw_frame = pd.read_csv(raw_train_csv)

print("Raw train csv:", raw_train_csv)
print("Raw columns:", list(raw_frame.columns))
print(raw_frame.head())

# APTOS official train.csv should have: id_code, diagnosis
if "id_code" not in raw_frame.columns:
    raise ValueError(
        f"APTOS train.csv should contain id_code, but got: {list(raw_frame.columns)}"
    )

if "diagnosis" not in raw_frame.columns:
    raise ValueError(
        f"APTOS train.csv should contain diagnosis, but got: {list(raw_frame.columns)}"
    )

# -----------------------------
# 3. Convert APTOS CSV to Jiaozi-friendly CSV
#    Original: id_code, diagnosis
#    New: image, label
# -----------------------------
processed_frame = raw_frame[["id_code", "diagnosis"]].copy()
processed_frame = processed_frame.rename(
    columns={
        "id_code": "image",
        "diagnosis": "label",
    }
)

# APTOS train images are named {id_code}.png
processed_frame["image"] = processed_frame["image"].astype(str) + ".png"
processed_frame["label"] = processed_frame["label"].astype(int)

processed_train_csv = raw_train_csv.with_name("jiaozi_train.csv")
processed_frame.to_csv(processed_train_csv, index=False)

print("\nProcessed train csv:", processed_train_csv)
print(processed_frame.head())
print("\nClass distribution:")
print(processed_frame["label"].value_counts().sort_index().to_string())

# -----------------------------
# 4. Locate train_images directory
# -----------------------------
candidate_image_dirs = [
    raw_train_csv.parent / "train_images",
    DATASET_DIR / "train_images",
]

image_dir = None
for d in candidate_image_dirs:
    if d.exists() and d.is_dir():
        image_dir = d
        break

if image_dir is None:
    matches = [p for p in DATASET_DIR.rglob("train_images") if p.is_dir()]
    if matches:
        image_dir = matches[0]

if image_dir is None:
    raise FileNotFoundError(f"Could not find train_images directory under {DATASET_DIR}")

print("\nImage dir:", image_dir)

# Check one sample image exists
sample_image = processed_frame.iloc[0]["image"]
sample_path = image_dir / sample_image

if not sample_path.exists():
    raise FileNotFoundError(
        f"Sample image not found: {sample_path}. "
        f"Please check whether train_images contains PNG files."
    )

print("Sample image exists:", sample_path)

# -----------------------------
# 5. Local data info
# -----------------------------
LOCAL_DATA_INFO = {
    "benchmark": "aptos_2019_kaggle",
    "competition": "aptos2019-blindness-detection",
    "train_csv": str(processed_train_csv),
    "image_dir": str(image_dir),
    "image_column": "image",
    "label_column": "label",
    "target_column": "label",
    "image_path_template": "{image}",
    "image_extension": "",
    "num_classes": 5,
    "metric": "qwk",
}

print("\nPrepared local Kaggle APTOS data:")
print(json.dumps(LOCAL_DATA_INFO, indent=2, ensure_ascii=False))

# Clean old output
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

# -----------------------------
# 6. Import Jiaozi modules
# -----------------------------
from analyzer.image_statistics import ImageStatisticsAnalyzer
from features_extraction_api import module1_pipeline
from pipeline import derive_recommended_epochs, merge_modules, run_module4_generation
from recommender import OutcomeMemory, recommend
from retrieval.rag_retrieval import (
    build_all_task_lists,
    build_graph,
    build_vector_index,
    print_results,
    retrieve_top3_hybrid,
)

# -----------------------------
# 7. Module 1
# -----------------------------
print("\n[Notebook] Module 1: Parsing user requirements...")
m1_output = module1_pipeline(QUERY)

if m1_output is None:
    raise RuntimeError("Module 1 failed, so no Module 3 or Module 4 output was produced.")

# -----------------------------
# 8. Local image dataset wrapper
# -----------------------------
class LocalImageSplit:
    column_names = ["image", "label"]

    def __init__(self, frame, info):
        from PIL import Image

        self._Image = Image
        self.labels = frame[info["label_column"]].tolist()
        self.image_paths = []

        image_root = Path(info["image_dir"])
        image_column = info["image_column"]
        label_column = info["label_column"]
        template = info.get("image_path_template", "{image}")
        extension = info.get("image_extension", "")

        for _, row in frame.iterrows():
            image_value = str(row[image_column])
            relative = template.format(
                image=image_value,
                label=str(row[label_column]),
                stem=Path(image_value).stem,
            )

            image_path = image_root / relative

            if extension and not image_path.suffix:
                image_path = image_path.with_suffix(extension)

            if not image_path.is_file():
                raise FileNotFoundError(
                    f"Training image referenced by CSV is missing: {image_path}"
                )

            self.image_paths.append(image_path)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, key):
        if key == "label":
            return self.labels

        if key == "image":
            return [self._open_image(path) for path in self.image_paths]

        index = int(key)
        return {
            "image": self._open_image(self.image_paths[index]),
            "label": self.labels[index],
        }

    def _open_image(self, path):
        with self._Image.open(path) as image:
            return image.convert("RGB")


def build_local_module2_dataset(info):
    frame = pd.read_csv(info["train_csv"])

    required = {info["image_column"], info["label_column"]}
    missing = required - set(frame.columns)

    if missing:
        raise ValueError(f"Missing required CSV columns: {sorted(missing)}")

    return {"train": LocalImageSplit(frame, info)}


# -----------------------------
# 9. Module 2
# -----------------------------
print("\n[Notebook] Module 2: Analyzing sampled real Kaggle APTOS images...")
m2_report = ImageStatisticsAnalyzer().analyze(
    build_local_module2_dataset(LOCAL_DATA_INFO)
)

# -----------------------------
# 10. Merge M1 + M2
# -----------------------------
m3_input = merge_modules(m1_output, m2_report)

# Force benchmark-specific metric
m3_input["evaluation_metric"] = "qwk"
m3_input["num_classes"] = 5

print(
    f"\n[Notebook] Merged: task={m3_input['task_type']} "
    f"size={m3_input['data_size']} "
    f"classes={m3_input.get('num_classes')} "
    f"metric={m3_input['evaluation_metric']}"
)

# -----------------------------
# 11. Module 3
# -----------------------------
print("\n[Notebook] Module 3: Retrieving model configurations...")
graph = build_graph()
collection = build_vector_index()
recommendations = retrieve_top3_hybrid(m3_input, graph, collection)

print_results(m3_input, recommendations, graph)

try:
    recommendations = recommend(
        recommendations,
        m2_report,
        m3_input,
        memory=OutcomeMemory(),
    )
    print("[Notebook] Recommender re-ranked candidates.")
except Exception as exc:
    print(f"[Notebook] Recommender skipped: {exc}")

# -----------------------------
# 12. Build task lists and inject APTOS settings
# -----------------------------
task_lists = build_all_task_lists(recommendations, graph, fmt="nl")

for task_list in task_lists:
    model_config = task_list.get("model_config")

    if not isinstance(model_config, dict):
        continue

    model_config.update(
        {
            "num_classes": LOCAL_DATA_INFO["num_classes"],
            "train_csv": LOCAL_DATA_INFO["train_csv"],
            "image_dir": LOCAL_DATA_INFO["image_dir"],
            "image_column": LOCAL_DATA_INFO["image_column"],
            "label_column": LOCAL_DATA_INFO["label_column"],
            "target_column": LOCAL_DATA_INFO["target_column"],
            "image_path_template": LOCAL_DATA_INFO["image_path_template"],
            "image_extension": LOCAL_DATA_INFO["image_extension"],
            "evaluation_metric": LOCAL_DATA_INFO["metric"],
            "metric": LOCAL_DATA_INFO["metric"],
            "metric_name": LOCAL_DATA_INFO["metric"],
            "offline_smoke": not REAL_TRAINING,
            "benchmark_key": LOCAL_DATA_INFO["benchmark"],
            "competition": LOCAL_DATA_INFO["competition"],
        }
    )

    model_config.setdefault(
        "recommended_epochs",
        derive_recommended_epochs(
            m3_input.get("data_size", "medium"),
            model_config.get("finetune_strategy"),
            bool(model_config.get("pretrained_hf_id")),
        ),
    )

print("\nTop model config preview:")
print(json.dumps(task_lists[0]["model_config"], indent=2)[:2500])

# -----------------------------
# 13. Module 4 code generation
# -----------------------------
print("\n[Notebook] Module 4: Generating real-training project...")

module4 = run_module4_generation(
    task_lists,
    OUTPUT_DIR,
    skip_smoke=REAL_TRAINING,
    timeout=120,
    llm_provider="openai",
)

summary_path = OUTPUT_DIR / "module4_summary.json"

if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    raise FileNotFoundError(
        f"Module 4 summary was not generated at {summary_path}. Available files: {available}"
    )

DATASET = LOCAL_DATA_INFO["train_csv"]

print("\nModule 4 summary:", summary_path)
print("Generated project output dir:", OUTPUT_DIR)
print("DATASET:", DATASET)


Current working directory: /content/Jiaozi
Using pipeline: /content/Jiaozi/pipeline.py
Raw train csv: /content/data/aptos/aptos2019-blindness-detection/train.csv
Raw columns: ['id_code', 'diagnosis']
        id_code  diagnosis
0  000c1434d8d7          2
1  001639a390f0          4
2  0024cdab0c1e          1
3  002c21358ce6          0
4  005b95c28852          0

Processed train csv: /content/data/aptos/aptos2019-blindness-detection/jiaozi_train.csv
              image  label
0  000c1434d8d7.png      2
1  001639a390f0.png      4
2  0024cdab0c1e.png      1
3  002c21358ce6.png      0
4  005b95c28852.png      0

Class distribution:
label
0    1805
1     370
2     999
3     193
4     295

Image dir: /content/data/aptos/aptos2019-blindness-detection/train_images
Sample image exists: /content/data/aptos/aptos2019-blindness-detection/train_images/000c1434d8d7.png

Prepared local Kaggle APTOS data:
{
  "benchmark": "aptos_2019_kaggle",
  "competition": "aptos2019-blindness-detection",
  "train_cs

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[Index] Added 18 backbone entries and refreshed 18 total entries.

──────────────────────────────────────────────────────────────────────
Task   : classification
Input  : size=medium  priority=accuracy
Flags  : class_imbalance, medical
Desc   : Diabetic retinopathy severity grading from retinal fundus photographs. Use label_code as the numeric target column for 5 ordered classes. Evaluate using Quadratic Weighted Kappa. This is a medical retinal image task, so the pipeline should include robust image preprocessing, such as resizing, normalization, color and brightness augmentation, and full fine-tuning of a pretrained model. Prefer EfficientNet-B0 or EfficientNet-B3 with ImageNet pretrained weights. Handle class imbalance and save the best checkpoint according to validation QWK.
──────────────────────────────────────────────────────────────────────

Top 1  [score=0.805  struct=1.0  vec=0.512]
  backbone  : DINOv3
  head      : Classification Head
  loss      : CrossEntropyLoss
  optimi

In [8]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')


{
  "candidates": [
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 1,
      "score": 0.805,
      "task_type": "classification"
    },
    {
      "backbone": "dinov3",
      "finetune_strategy": "partial",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 2,
      "score": 0.805,
      "task_type": "classification"
    },
    {
      "backbone": "dinov2",
      "finetune_strategy": "full",
      "loss": "cross_entropy_loss",
      "optimizer": "adamw",
      "rank": 3,
      "score": 0.722,
      "task_type": "classification"
    },
    {
      "backbone": "swin_transformer",
      "finetune_strategy": "full",
      "loss": "focal_loss",
      "optimizer": "adamw",
      "rank": 4,
      "score": 0.718,
      "task_type": "classification"
    }
  ],
  "errors": [],
  "generated_files": [
    "README_generated.md",
    "configs.json",
    "evaluate.py",
   

## Train the recommended model (real data)

This version avoids accidentally reusing old checkpoints when you change the query. It streams training logs line by line, so epoch results should appear during training.


In [9]:
# [Add New] Patch train.py to force epoch-level print flushing

from pathlib import Path
import re

train_path = Path("/content/jiaozi_generated_pipeline/train.py")
text = train_path.read_text(encoding="utf-8")

# Whenever possible, change standard print(...) calls to print(..., flush=True)
# to prevent Colab's buffering from delaying output until training is complete.
text = re.sub(
    r"print\((f?['\"]\[train\].*?)\)",
    r"print(\1, flush=True)",
    text,
)

train_path.write_text(text, encoding="utf-8")
print("Patched train.py print statements with flush=True.")


Patched train.py print statements with flush=True.


In [10]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

# ========= Training controls =========
# After modifying the query, it is recommended to use a different name each time; this prevents loading an old checkpoint.
RUN_NAME = "aptos_query_v2"

# True = Train from scratch, deleting any existing checkpoint with the same name; False = Attempt to automatically resume training.
FORCE_FRESH_TRAINING = True

# True = Save to Google Drive; False = Save only to the current Colab runtime.
SAVE_CHECKPOINTS_TO_DRIVE = True

# "None" indicates using the EPOCHS setting from Cell 11; alternatively, you can directly specify 1, 3, or 20.
EPOCHS_OVERRIDE = None
# ====================================

if not REAL_TRAINING:
    print("REAL_TRAINING is False - skipping the real training step.")
    print("Set REAL_TRAINING = True in the run cell above to train on real data.")
else:
    cfg_path = OUTPUT_DIR / "configs.json"
    configs = json.loads(cfg_path.read_text(encoding="utf-8"))
    cfg = configs[0]
    mc = cfg.get("model_config", cfg)

    epochs = EPOCHS if EPOCHS_OVERRIDE is None else EPOCHS_OVERRIDE
    if epochs is None:
        epochs = mc.get("recommended_epochs") or cfg.get("recommended_epochs") or 10

    backbone = mc.get("backbone") or cfg.get("backbone") or "unknown_backbone"
    dataset_used = mc.get("dataset_id") or cfg.get("dataset_id") or DATASET

    # Record the query associated with this experiment to facilitate verifying later whether it is a "new query."
    cfg["experiment_query"] = QUERY
    cfg["run_name"] = RUN_NAME

    if SAVE_CHECKPOINTS_TO_DRIVE and Path("/content/drive/MyDrive").exists():
        # Stop using the dataset CSV path as a tag—it makes it too easy to reuse old directories; use `RUN_NAME` to clearly distinguish between experiments.
        tag = f"{RUN_NAME}_{backbone}"
        ckpt_dir = f"/content/drive/MyDrive/Jiaozi/checkpoints/{tag}"

        if FORCE_FRESH_TRAINING and Path(ckpt_dir).exists():
            print("Removing old checkpoint dir:", ckpt_dir)
            shutil.rmtree(ckpt_dir)

        os.makedirs(ckpt_dir, exist_ok=True)

        cfg["checkpoint_dir"] = ckpt_dir
        cfg["resume_checkpoint"] = None if FORCE_FRESH_TRAINING else "auto"

        cfg_path.write_text(
            json.dumps(configs, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        print(
            "Checkpoints ->",
            ckpt_dir,
            "(resume:",
            "disabled, fresh training" if FORCE_FRESH_TRAINING else "auto",
            ")",
        )

    elif SAVE_CHECKPOINTS_TO_DRIVE:
        print("WARNING: Drive not mounted; checkpoints will be ephemeral.")
        cfg["resume_checkpoint"] = None if FORCE_FRESH_TRAINING else "auto"
        cfg_path.write_text(
            json.dumps(configs, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

    print("Experiment RUN_NAME:", RUN_NAME)
    print("Query used:")
    print(QUERY)
    print(f"\nTraining {backbone} for {epochs} epochs on {dataset_used} ...")

    train_command = [sys.executable, "-u", "run.py", "--epochs", str(epochs)]
    print("Command:", " ".join(train_command), "(cwd:", OUTPUT_DIR, ")\n")

    # Key point: Use `Popen` to print line-by-line in real-time, rather than using `stdout=PIPE` and waiting for the process to finish completely before printing.
    process = subprocess.Popen(
        train_command,
        cwd=str(OUTPUT_DIR),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training failed with return code {return_code}")

    print("\nReal training finished. Checkpoints under:", cfg.get("checkpoint_dir", "checkpoints (ephemeral)"))


Checkpoints -> /content/drive/MyDrive/Jiaozi/checkpoints/aptos_query_v2_dinov3 (resume: disabled, fresh training )
Experiment RUN_NAME: aptos_query_v2
Query used:
Diabetic retinopathy severity grading from retinal fundus photographs. Use label_code as the numeric target column for 5 ordered classes. Evaluate using Quadratic Weighted Kappa. This is a medical retinal image task, so the pipeline should include robust image preprocessing, such as resizing, normalization, color and brightness augmentation, and full fine-tuning of a pretrained model. Prefer EfficientNet-B0 or EfficientNet-B3 with ImageNet pretrained weights. Handle class imbalance and save the best checkpoint according to validation QWK.

Training dinov3 for 20 epochs on /content/data/aptos/aptos2019-blindness-detection/jiaozi_train.csv ...
Command: /usr/bin/python3 -u run.py --epochs 20 (cwd: /content/jiaozi_generated_pipeline )


Loading weights: 100%|██████████| 211/211 [00:07<00:00, 26.64it/s]
[train] requested_backbone=

## Show the result

Reads the best checkpoint and prints its validation score **instantly** (the metric was
already computed during training — no re-evaluation). Set `FULL_EVAL = True` for a full
re-evaluation (macro-F1, params, sample count), which costs one extra pass over the eval set.


In [11]:
import json, torch
from pathlib import Path

# Locate best_model.pt (prefer the Drive checkpoint dir set by the training cell).
_candidates = []
try:
    _candidates.append(Path(ckpt_dir) / 'best_model.pt')
except NameError:
    pass
_candidates.append(Path(OUTPUT_DIR) / 'checkpoints' / 'best_model.pt')
best_path = next((p for p in _candidates if p.exists()), None)
if best_path is None:
    raise FileNotFoundError(f'best_model.pt not found in: {[str(p) for p in _candidates]}')

ckpt = torch.load(best_path, map_location='cpu', weights_only=False)
print('=== RESULT (best checkpoint) ===')
print('checkpoint :', best_path)
print('best_epoch :', ckpt.get('best_epoch'))
print('best_metric:', ckpt.get('best_metric'), '(validation metric at best epoch — no re-eval)')
print('validation :', json.dumps(ckpt.get('validation'), ensure_ascii=False))

# Full re-evaluation (macro_f1, params, num_samples) — one extra pass over the eval set.
FULL_EVAL = False
if FULL_EVAL:
    import os, sys
    os.chdir(OUTPUT_DIR); sys.path.insert(0, str(OUTPUT_DIR))
    from model import build_model
    from evaluate import evaluate
    _cfg = json.loads((Path(OUTPUT_DIR) / 'configs.json').read_text(encoding='utf-8'))[0]
    _cfg.update(_cfg.pop('model_config', {}) or {})
    _model = build_model(_cfg); _model.load_state_dict(ckpt['model_state_dict'])
    print('\n=== FULL EVALUATE ===')
    print(json.dumps(evaluate(_model, _cfg), indent=2, ensure_ascii=False))


=== RESULT (best checkpoint) ===
checkpoint : /content/drive/MyDrive/Jiaozi/checkpoints/aptos_query_v2_dinov3/best_model.pt
best_epoch : 14
best_metric: 0.8916122057169453 (validation metric at best epoch — no re-eval)
validation : {"metric_name": "qwk", "metric_value": 0.8916122057169453, "accuracy": 0.8185538649559021, "epoch": 14}


In [12]:
# [replace Cell 17] Generate Kaggle APTOS submission.csv

import os
import sys
import json
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

OUTPUT_DIR = Path("/content/jiaozi_generated_pipeline")
DATA_DIR = Path("/content/data/aptos/aptos2019-blindness-detection")

sample_path = DATA_DIR / "sample_submission.csv"
test_csv = DATA_DIR / "test.csv"
image_dir = DATA_DIR / "test_images"
submission_path = OUTPUT_DIR / "submission.csv"

assert OUTPUT_DIR.exists(), OUTPUT_DIR
assert sample_path.exists(), sample_path
assert test_csv.exists(), test_csv
assert image_dir.exists(), image_dir

os.chdir(OUTPUT_DIR)
if str(OUTPUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUTPUT_DIR))

from model import build_model
from train import _build_image_transform

# -----------------------------
# 1. Load config
# -----------------------------
configs = json.loads((OUTPUT_DIR / "configs.json").read_text(encoding="utf-8"))
cfg = configs[0]

if "model_config" in cfg:
    flat_cfg = {**cfg, **cfg["model_config"]}
else:
    flat_cfg = cfg

# Force APTOS-related settings
flat_cfg["num_classes"] = 5
flat_cfg["metric_name"] = "qwk"
flat_cfg["evaluation_metric"] = "qwk"

print("Config preview:")
for k in ["backbone", "pretrained_hf_id", "model_name", "num_classes", "metric_name"]:
    if k in flat_cfg:
        print(k, ":", flat_cfg[k])

# -----------------------------
# 2. Find best checkpoint
# -----------------------------
ckpt_candidates = []

try:
    ckpt_candidates.append(Path(ckpt_dir) / "best_model.pt")
except NameError:
    pass

if flat_cfg.get("checkpoint_dir"):
    ckpt_candidates.append(Path(flat_cfg["checkpoint_dir"]) / "best_model.pt")

ckpt_candidates.append(OUTPUT_DIR / "checkpoints" / "best_model.pt")
ckpt_candidates.append(OUTPUT_DIR / "best_model.pt")

best_path = next((p for p in ckpt_candidates if p.exists()), None)

if best_path is None:
    raise FileNotFoundError(
        "best_model.pt not found. Checked:\n" + "\n".join(str(p) for p in ckpt_candidates)
    )

print("Using checkpoint:", best_path)

# -----------------------------
# 3. Load model
# -----------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = build_model(flat_cfg).to(device)

ckpt = torch.load(best_path, map_location=device, weights_only=False)

if "model_state_dict" in ckpt:
    state_dict = ckpt["model_state_dict"]
elif "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

model.load_state_dict(state_dict)
model.eval()

# -----------------------------
# 4. Build test transform
# -----------------------------
transform = _build_image_transform(flat_cfg, "test")

# -----------------------------
# 5. Read APTOS test ids
# -----------------------------
sample = pd.read_csv(sample_path)
test_df = pd.read_csv(test_csv)

print("sample_submission columns:", list(sample.columns))
print("test.csv columns:", list(test_df.columns))

assert "id_code" in sample.columns, sample.columns
assert "id_code" in test_df.columns, test_df.columns

test_ids = sample["id_code"].astype(str).tolist()

# -----------------------------
# 6. Predict
# -----------------------------
rows = []
batch = []
batch_ids = []

def logits_to_prediction(logits):
    """
    APTOS is ordinal 5-class grading.
    For Kaggle submission, diagnosis must be an integer from 0 to 4.

    Two common choices:
    1. argmax class
    2. expected value of class probabilities, rounded to nearest integer

    Here we use expected-value rounding because QWK is ordinal.
    """
    probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
    class_values = np.arange(5)
    pred_float = (probs * class_values).sum(axis=1)
    pred = np.rint(pred_float).astype(int)
    pred = np.clip(pred, 0, 4)
    return pred

def flush():
    global batch, batch_ids, rows

    if not batch:
        return

    x = torch.stack(batch).to(device)

    with torch.no_grad():
        out = model(x)

        # Some models return tuple/list/dict
        if isinstance(out, dict):
            logits = out.get("logits", next(iter(out.values())))
        elif isinstance(out, (tuple, list)):
            logits = out[0]
        else:
            logits = out

        pred = logits_to_prediction(logits)

    for image_id, diagnosis in zip(batch_ids, pred):
        rows.append(
            {
                "id_code": image_id,
                "diagnosis": int(diagnosis),
            }
        )

    batch = []
    batch_ids = []

for image_id in test_ids:
    img_path = image_dir / f"{image_id}.png"

    if not img_path.exists():
        raise FileNotFoundError(img_path)

    img = Image.open(img_path).convert("RGB")
    batch.append(transform(img))
    batch_ids.append(image_id)

    if len(batch) >= 64:
        flush()

flush()

# -----------------------------
# 7. Save submission
# -----------------------------
sub = pd.DataFrame(rows)

# Keep the same order as sample_submission
sub = sample[["id_code"]].merge(sub, on="id_code", how="left")

# Sanity checks
assert len(sub) == len(sample), f"submission rows {len(sub)} != sample rows {len(sample)}"
assert sub["diagnosis"].notna().all(), "Some predictions are missing."
assert sub["diagnosis"].between(0, 4).all(), "Predictions must be in [0, 4]."

sub["diagnosis"] = sub["diagnosis"].astype(int)

sub.to_csv(submission_path, index=False)

print("Wrote submission:", submission_path)
print("Shape:", sub.shape)
print("Prediction distribution:")
print(sub["diagnosis"].value_counts().sort_index())
display(sub.head())


Config preview:
backbone : dinov3
pretrained_hf_id : facebook/dinov3-vitb16-pretrain-lvd1689m
num_classes : 5
metric_name : qwk
Using checkpoint: /content/drive/MyDrive/Jiaozi/checkpoints/aptos_query_v2_dinov3/best_model.pt


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

sample_submission columns: ['id_code', 'diagnosis']
test.csv columns: ['id_code']
Wrote submission: /content/jiaozi_generated_pipeline/submission.csv
Shape: (1928, 2)
Prediction distribution:
diagnosis
0     222
1     264
2    1115
3     298
4      29
Name: count, dtype: int64


,id_code,diagnosis
0,0005cfc8afb6,2
1,003f0afdcd15,3
2,006efc72b638,2
3,00836aaacf06,2
4,009245722fa4,3


In [13]:
!kaggle competitions submit \
  -c aptos2019-blindness-detection \
  -f /content/jiaozi_generated_pipeline/submission.csv \
  -m "Jiaozi APTOS late submission"


100% 28.3k/28.3k [00:00<00:00, 41.8kB/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission


In [14]:
!kaggle competitions submissions -c aptos2019-blindness-detection


No submissions found


In [15]:
!kaggle --version


Kaggle CLI 2.2.3


In [16]:
from pathlib import Path

samples = list(Path("/content").rglob("sample_submission.csv"))

print("Found sample_submission：")
for s in samples:
    print(s)


找到的 sample_submission：
/content/data/aptos/aptos2019-blindness-detection/sample_submission.csv


In [17]:
!kaggle competitions submit \
    -c aptos2019-blindness-detection \
    -f /content/data/aptos/aptos2019-blindness-detection/sample_submission.csv\
    -m "Official sample submission test"


100% 28.3k/28.3k [00:00<00:00, 42.7kB/s]
400 Client Error: Bad Request for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/CreateSubmission
